# Figure 2 — Between-language Over/Underexpression (Heatworm)

This figure shows whether each language’s weekly Ukraine-related usage is above or below its expected share of total cross-language usage.

- **Color**: underexpression (blue) → near expected share (gray) → overexpression (pink/red)
- **Marker height**: raw weekly frequency

## Input
- `config.CHOSEN_WEEKLY_PIVOT_FILE` (`data/processed/chosen_words_weekly_pivoted.csv`)

## Outputs
- `outputs/figures/Fig.2_between-language/` (heatworm: `pdf`, `svg`, `jpeg`, `html`, optional `eps`)
- `outputs/figures/Fig.2_between-language/distribution/` (distribution plot in same formats)

**Prerequisite:** run `02_combine_data.ipynb` first.

In [6]:
import sys
sys.path.insert(0, '..')

# Force config reload (Jupyter caches imports)
if 'config' in sys.modules:
    del sys.modules['config']

from config import CHOSEN_WEEKLY_PIVOT_FILE, WORD_FORMS_ALL, FIGURES_DIR

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
import subprocess

pio.templates.default = "plotly_white"

## ⚙️ Parameters

**Only edit this cell** to change the plot.  All subsequent cells run as-is.

In [7]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  Edit these variables — everything below runs automatically.    │
# └─────────────────────────────────────────────────────────────────┘

remove_UA_RU = False   # True  → drop Ukrainian & Russian from the plot

smooth       = True    # True  → apply rolling-window smoothing
smooth_time  = 6       # smoothing window in weeks (only active when smooth=True)

# Save behavior controls
FORCE_OVERWRITE = True       # delete existing file before writing
APPEND_TIMESTAMP = True      # True => creates a new file each run

# Language display order — first entry appears at the TOP of the chart.
# Modify to change the row order without touching any other cell.
LANGUAGE_ORDER = [
    'Ukrainian', 'Russian', 'Arabic', 'Portuguese', 'Catalan', 'Korean',
    'Persian', 'Turkish', 'Indonesian', 'Urdu', 'Vietnamese',
    'Serbian', 'Estonian', 'Romanian', 'Greek', 'Hungarian', 'Polish',
    'Swedish', 'Czech', 'Spanish', 'Danish', 'English', 'Dutch',
    'Norwegian', 'Finnish', 'French', 'Italian', 'German',
]

# Key historical events: shown as vertical dotted lines on the x-axis.
EVENT_DATES = ['2009-01-05', '2012-06-08', '2014-02-27', '2022-02-24']

# Figure pixel dimensions (2000×1350 → ~180×122 mm at 283 DPI)
FIG_WIDTH  = 2000
FIG_HEIGHT = 1350

## Load & compute expression

In [8]:
# ── 1. Load pivot: language_ISO rows × weekly date columns ───────────────────
DFfreq = pd.read_csv(CHOSEN_WEEKLY_PIVOT_FILE, index_col=0)

# Robust index handling: supports both ISO-index and language-index pivots
meta_lang = pd.read_csv(WORD_FORMS_ALL, usecols=['ISO', 'Language']).drop_duplicates()
iso_to_name = meta_lang.set_index('ISO')['Language'].to_dict()
iso_set = set(meta_lang['ISO'])
name_set = set(meta_lang['Language'])

idx = pd.Index(DFfreq.index.astype(str))
if idx.isin(iso_set).all():
    DFfreq.index = idx.map(iso_to_name)
elif idx.isin(name_set).all():
    DFfreq.index = idx
else:
    DFfreq.index = idx.map(iso_to_name).fillna(idx)
DFfreq.index = pd.Index(DFfreq.index, name='language')

# ── 2. Optionally drop Ukrainian and Russian ──────────────────────────────────
if remove_UA_RU:
    DFfreq = DFfreq.drop(index=['Ukrainian', 'Russian'], errors='ignore')

# ── 3. Between-language over/underexpression ──────────────────────────────────
# Each week: total Ukraine-frequency across all languages
SumsFreq = DFfreq.sum().replace(0, np.nan)

# Each language's share of total weekly usage
DFfreqShare = DFfreq.div(SumsFreq, axis=1)

# Each language's average share (their "expected" proportion)
freqRowMeans = DFfreq.mean(axis=1)
safe_total_mean = freqRowMeans.sum()
freqShareRowMeans_Proportion = (
    freqRowMeans / safe_total_mean if safe_total_mean != 0 else freqRowMeans * np.nan
)

# Deviation from expected share: positive = above expected, negative = below
DFresultNonLog = DFfreqShare.sub(freqShareRowMeans_Proportion, axis=0)
DFresultNonLog = DFresultNonLog.replace([np.inf, -np.inf], np.nan)

# Signed-log scale (same convention as within-language)
DFresultLog = DFresultNonLog.map(
    lambda x: 0 if pd.isna(x) or x == 0 else (np.log10(x * 100_000) if x > 0 else -np.log10(-x * 100_000))
)

# ── 4. Optional temporal smoothing ───────────────────────────────────────────
if smooth:
    DFresultLog    = DFresultLog.T.rolling(window=smooth_time, min_periods=1).mean().T
    DFresultNonLog = DFresultNonLog.T.rolling(window=smooth_time, min_periods=1).mean().T
    DFfreq         = DFfreq.T.rolling(window=smooth_time, min_periods=1).mean().T

# ── 5. Reshape wide → long for Plotly ────────────────────────────────────────
AttentionLog    = DFresultLog.reset_index().melt(
                    id_vars=['language'], var_name='Date', value_name='ExpressionLog')
AttentionNonLog = DFresultNonLog.reset_index().melt(
                    id_vars=['language'], var_name='Date', value_name='Expression')
Freq            = DFfreq.reset_index().melt(
                    id_vars=['language'], var_name='Date', value_name='FractionRaw')

merged_df = (Freq
             .merge(AttentionLog,    on=['language', 'Date'], how='inner')
             .merge(AttentionNonLog, on=['language', 'Date'], how='inner'))

# Marker size = log-shifted raw frequency (must stay positive for Plotly sizing)
merged_df['Fraction']    = merged_df['FractionRaw'].replace(0, np.nan)
merged_df['FractionLog'] = (np.log10(merged_df['Fraction']) + 7.5).fillna(0)
merged_df['FractionRaw'] = merged_df['FractionRaw'].fillna(0)

# Apply display order (first item → top of y-axis)
display_order = list(reversed(LANGUAGE_ORDER))
merged_df['language'] = pd.Categorical(merged_df['language'],
                                        categories=display_order, ordered=True)
merged_df = merged_df.sort_values(['language', 'Date'])

print(f"Loaded {merged_df['language'].nunique()} languages "
      f"× {merged_df['Date'].nunique()} weeks")

Loaded 28 languages × 769 weeks


## Colorscale

In [9]:
# Dynamic colorscale for between-language expression.
# The between-language signal has a narrower dynamic range than within-language,
# so the neutral (gray) zone covers just −3 to +3.
#
# Color key: same as Fig.1/S2 (blue = under, gray = neutral, pink/red = over)

stats = merged_df['ExpressionLog'].describe()
min_val, max_val = stats['min'], stats['max']

color_points = [
    (min_val, '#013b50'),  # extreme underexpression – dark blue
    (-3,      '#1b83a9'),  # strong underexpression  – light blue
    ( 0,      '#B6B6A8'),  # at mean                 – gray
    ( 3,      '#a41a6b'),  # strong overexpression   – deep pink
    (max_val, '#2a0313'),  # extreme overexpression  – dark red
]

custom_colorscale = [
    [(v - min_val) / (max_val - min_val), color]
    for v, color in color_points
]
print(f"ExpressionLog range: {min_val:.2f} to {max_val:.2f}")

ExpressionLog range: -4.43 to 4.68


## Heatworm plot

Scatter figure where:
- **x** = week, **y** = language
- **marker height** = log-scaled raw frequency (how many tweets that week)
- **color** = signed-log expression (how far above/below the language's mean)

In [10]:
n_languages = merged_df['language'].nunique()
setheight   = 26 if remove_UA_RU else 28   # y-limit for vertical event lines

# ── Marker height: auto-scale to fill ~52 % of each language row ─────────────
row_height_px    = FIG_HEIGHT / n_languages
computed_sizeref = float(merged_df['FractionLog'].max() / (row_height_px * 0.52))

# ── Bar width: fill each weekly slot exactly ──────────────────────────────────
n_slots           = pd.to_datetime(merged_df['Date']).nunique()
target_line_width = FIG_WIDTH / n_slots

# ── Hover text ────────────────────────────────────────────────────────────────
hovertext = [
    f"Fraction: {fr}<br>ExpressionLog: {el:.2f}"
    for fr, el in zip(merged_df['Expression'], merged_df['ExpressionLog'])
]

# ── Build scatter figure ──────────────────────────────────────────────────────
fig = go.Figure(data=[go.Scatter(
    x=merged_df['Date'],
    y=merged_df['language'],
    mode='markers',
    hovertext=hovertext,
    marker=dict(
        symbol='line-ns',
        sizemode='diameter',
        sizeref=computed_sizeref,
        sizemin=1,
        size=merged_df['FractionLog'],
        color=merged_df['ExpressionLog'],
        opacity=1,
        colorscale=custom_colorscale,
            cmax=4.5,
        line=dict(
            color=merged_df['ExpressionLog'],
            colorscale=custom_colorscale,
            width=target_line_width,
        ),
        colorbar=dict(
            orientation='h', thickness=50, len=0.6,
            x=0.5, xanchor='center', y=-0.3,
            tickvals=[-4, -3, -1, 0, 1, 3, 4],
            ticktext=['-10%', '-1%', '-0.01%', '0%', '+0.01%', '+1%', '+10%'],

            ticks='outside', tickangle=0,
        ),
    ),
)])

# ── Vertical event lines ──────────────────────────────────────────────────────
for date in EVENT_DATES:
    fig.add_shape(
        type='line', x0=date, x1=date, y0=-2, y1=setheight,
        line=dict(color='black', width=0.6, dash='dot'),
    )

# ── Axes and layout ───────────────────────────────────────────────────────────
yearly_ticks = pd.date_range(start='2008-01-01', end='2023-12-31', freq='YS')
tick_text    = [str(d.year) if d.year >= 2009 else '' for d in yearly_ticks]

fig.update_layout(
    width=FIG_WIDTH, height=FIG_HEIGHT,
    margin=dict(l=0, r=10, t=0, b=0),
    font=dict(size=32),
    xaxis=dict(
        tickmode='array',
        tickvals=yearly_ticks.strftime('%Y-%m-%d').tolist(),
        ticktext=tick_text,
        tickangle=270,
        range=['2008-10-01', '2023-05-30'],
        showgrid=True, gridcolor='rgba(175,175,175,0.2)',
    ),
    yaxis=dict(
        tickmode='linear', dtick=1,
        range=[-1, 30],
        showgrid=True, gridcolor='rgba(175,175,175,0.2)',
    ),
    xaxis_title='', yaxis_title='',
)

fig.show()

## Save heatworm

Saves to `outputs/figures/` (subfolders created automatically).
EPS tries kaleido native writer, then Ghostscript (`gs` / `gswin64c`), then `pdf2ps`.

In [11]:
# ── Output paths ──────────────────────────────────────────────────────────────
smooth_tag = f'Smooth{smooth_time}w' if smooth else 'NoSmooth'
ua_ru_tag  = 'NoUA-RU' if remove_UA_RU else 'withUA-RU'
base_fig_name = f'Heatworm_{smooth_tag}_{ua_ru_tag}'
run_tag = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
fig_name = f'{base_fig_name}_{run_tag}' if APPEND_TIMESTAMP else base_fig_name

fig_dir = FIGURES_DIR / 'Fig.2_between-language'

print(f'Smoothing enabled: {smooth} (window={smooth_time} weeks)')
print(f'Saving base directory: {fig_dir.resolve()}')

# ── pdf · svg · jpeg · html ───────────────────────────────────────────────────
for fmt in ['pdf', 'svg', 'jpeg', 'html']:
    out_dir = fig_dir / fmt
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{fig_name}.{fmt}'
    if FORCE_OVERWRITE and out_path.exists():
        out_path.unlink()
    if fmt == 'html':
        fig.write_html(str(out_path))
    else:
        fig.write_image(str(out_path), format=fmt, engine='kaleido')
    mtime = pd.to_datetime(out_path.stat().st_mtime, unit='s')
    print(f'✓  {fmt.upper():<5}  {out_path}  (modified: {mtime})')

# ── EPS ───────────────────────────────────────────────────────────────────────
# Tries three methods in order:
#   1. kaleido native EPS writer (works with kaleido 0.2.x)
#   2. Ghostscript gs/gswin64c/gswin32c  (PDF → EPS)
#   3. pdf2ps (alternative Ghostscript front-end)
eps_ok   = False
eps_dir  = fig_dir / 'eps'
eps_dir.mkdir(parents=True, exist_ok=True)
eps_path = eps_dir / f'{fig_name}.eps'

try:
    fig.write_image(str(eps_path), format='eps', engine='kaleido')
    eps_ok = True
    print(f'✓  EPS    {eps_path}')
except Exception:
    pass

if not eps_ok:
    pdf_path = fig_dir / 'pdf' / f'{fig_name}.pdf'
    for gs in ['gs', 'gswin64c', 'gswin32c']:   # tries Linux/Mac then Windows Ghostscript
        try:
            subprocess.run(
                [gs, '-dNOPAUSE', '-dBATCH', '-dEPSCrop',
                 '-sDEVICE=eps2write', f'-sOutputFile={eps_path}', str(pdf_path)],
                check=True, capture_output=True)
            eps_ok = True
            print(f'✓  EPS (via {gs})  {eps_path}')
            break
        except (FileNotFoundError, subprocess.CalledProcessError):
            continue

if not eps_ok:
    pdf_path = fig_dir / 'pdf' / f'{fig_name}.pdf'
    try:
        subprocess.run(['pdf2ps', str(pdf_path), str(eps_path)],
                       check=True, capture_output=True)
        eps_ok = True
        print(f'✓  EPS (via pdf2ps)  {eps_path}')
    except (FileNotFoundError, subprocess.CalledProcessError):
        pass

if not eps_ok:
    print('⚠  EPS skipped.')
    print('   Install Ghostscript: https://www.ghostscript.com/releases/gsdnld.html')
    print('   Then re-run this save cell.')

Smoothing enabled: True (window=6 weeks)
Saving base directory: C:\Users\markm\Documents\Programming\2022_Ukraine-Twitter\pipeline\outputs\figures\Fig.2_between-language
✓  PDF    C:\Users\markm\Documents\Programming\2022_Ukraine-Twitter\pipeline\outputs\figures\Fig.2_between-language\pdf\Heatworm_Smooth6w_withUA-RU_20260401_002946.pdf  (modified: 2026-03-31 21:29:52.769427061)
✓  PDF    C:\Users\markm\Documents\Programming\2022_Ukraine-Twitter\pipeline\outputs\figures\Fig.2_between-language\pdf\Heatworm_Smooth6w_withUA-RU_20260401_002946.pdf  (modified: 2026-03-31 21:29:52.769427061)
✓  SVG    C:\Users\markm\Documents\Programming\2022_Ukraine-Twitter\pipeline\outputs\figures\Fig.2_between-language\svg\Heatworm_Smooth6w_withUA-RU_20260401_002946.svg  (modified: 2026-03-31 21:29:56.528156996)
✓  SVG    C:\Users\markm\Documents\Programming\2022_Ukraine-Twitter\pipeline\outputs\figures\Fig.2_between-language\svg\Heatworm_Smooth6w_withUA-RU_20260401_002946.svg  (modified: 2026-03-31 21:29:

## Distribution of expression values

Density histogram of all ExpressionLog values (all languages × all weeks).
Characterises the overall distribution and justifies the signed-log color scale.

In [12]:
# Distribution of ExpressionLog values across all language × week observations.
# The shape characterises the overall bias towards over- or underexpression
# and justifies the signed-log color scale used in the heatworm.

all_values = DFresultLog.values.flatten()  # one value per language × week

fig_dist = px.histogram(
    all_values,
    nbins=170,
    histnorm='density',
    color_discrete_sequence=['gray'],
)
fig_dist.update_traces(marker_line_width=0)
fig_dist.update_layout(
    xaxis_title='',
    yaxis_title='',
    bargap=0,
    bargroupgap=0,
    font=dict(size=28),
    template='simple_white',
    showlegend=False,
    margin=dict(l=50, r=50, t=60, b=50),
    width=2127,   # 180 mm at 300 DPI
    height=700,
)

fig_dist.show()

## Save distribution

In [13]:
dist_dir = FIGURES_DIR / 'Fig.2_between-language' / 'distribution'

for fmt in ['pdf', 'svg', 'jpeg', 'html']:
    out_dir = dist_dir / fmt
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'Distribution_{fig_name}.{fmt}'
    if FORCE_OVERWRITE and out_path.exists():
        out_path.unlink()
    if fmt == 'html':
        fig_dist.write_html(str(out_path))
    else:
        fig_dist.write_image(str(out_path), format=fmt, engine='kaleido')
    mtime = pd.to_datetime(out_path.stat().st_mtime, unit='s')
    print(f'✓  {fmt.upper():<5}  {out_path}  (modified: {mtime})')

# ── EPS ───────────────────────────────────────────────────────────────────────
# Tries three methods in order:
#   1. kaleido native EPS writer (works with kaleido 0.2.x)
#   2. Ghostscript gs/gswin64c/gswin32c  (PDF → EPS)
#   3. pdf2ps (alternative Ghostscript front-end)
eps_ok   = False
eps_dir  = dist_dir / 'eps'
eps_dir.mkdir(parents=True, exist_ok=True)
eps_path = eps_dir / f'{fig_name}.eps'

try:
    fig_dist.write_image(str(eps_path), format='eps', engine='kaleido')
    eps_ok = True
    print(f'✓  EPS    {eps_path}')
except Exception:
    pass

if not eps_ok:
    pdf_path = dist_dir / 'pdf' / f'{fig_name}.pdf'
    for gs in ['gs', 'gswin64c', 'gswin32c']:   # tries Linux/Mac then Windows Ghostscript
        try:
            subprocess.run(
                [gs, '-dNOPAUSE', '-dBATCH', '-dEPSCrop',
                 '-sDEVICE=eps2write', f'-sOutputFile={eps_path}', str(pdf_path)],
                check=True, capture_output=True)
            eps_ok = True
            print(f'✓  EPS (via {gs})  {eps_path}')
            break
        except (FileNotFoundError, subprocess.CalledProcessError):
            continue

if not eps_ok:
    pdf_path = dist_dir / 'pdf' / f'{fig_name}.pdf'
    try:
        subprocess.run(['pdf2ps', str(pdf_path), str(eps_path)],
                       check=True, capture_output=True)
        eps_ok = True
        print(f'✓  EPS (via pdf2ps)  {eps_path}')
    except (FileNotFoundError, subprocess.CalledProcessError):
        pass

if not eps_ok:
    print('⚠  EPS skipped.')
    print('   Install Ghostscript: https://www.ghostscript.com/releases/gsdnld.html')
    print('   Then re-run this save cell.')

✓  PDF    C:\Users\markm\Documents\Programming\2022_Ukraine-Twitter\pipeline\outputs\figures\Fig.2_between-language\distribution\pdf\Distribution_Heatworm_Smooth6w_withUA-RU_20260401_002946.pdf  (modified: 2026-03-31 21:30:20.627258301)
✓  SVG    C:\Users\markm\Documents\Programming\2022_Ukraine-Twitter\pipeline\outputs\figures\Fig.2_between-language\distribution\svg\Distribution_Heatworm_Smooth6w_withUA-RU_20260401_002946.svg  (modified: 2026-03-31 21:30:20.767390490)
✓  JPEG   C:\Users\markm\Documents\Programming\2022_Ukraine-Twitter\pipeline\outputs\figures\Fig.2_between-language\distribution\jpeg\Distribution_Heatworm_Smooth6w_withUA-RU_20260401_002946.jpeg  (modified: 2026-03-31 21:30:20.954311609)
✓  HTML   C:\Users\markm\Documents\Programming\2022_Ukraine-Twitter\pipeline\outputs\figures\Fig.2_between-language\distribution\html\Distribution_Heatworm_Smooth6w_withUA-RU_20260401_002946.html  (modified: 2026-03-31 21:30:20.985192776)
✓  JPEG   C:\Users\markm\Documents\Programming\2